## 2.1 理论计算题 — 卷积层计算

输入：3 × 32 × 32，16个卷积核 3×5×5，padding=2，stride=2

---

### 1. 输出特征图尺寸

公式：H_out = floor((H_in + 2P - K) / S) + 1

计算：
H_out = floor((32 + 4 - 5) / 2) + 1 = floor(31/2) + 1 = 15 + 1 = 16
W_out = floor((32 + 4 - 5) / 2) + 1 = 16
C_out = 16（卷积核个数）

答案：输出特征图尺寸为 16 × 16 × 16

---

### 2. 单个输出像素的乘法次数

每个卷积核覆盖输入的所有3个通道，空间尺寸为 5×5。
单个卷积核的权重数量 = 3 × 5 × 5 = 75

每个输出像素值 = 输入像素与这75个权重逐一相乘再求和
因此需要 75 次乘法

答案：75次点乘（乘法）操作

In [3]:
import numpy as np

def max_pool2d(X, pool_size, stride=None, padding=0):
    """
    手动实现二维最大池化前向传播
    """
    # 统一 pool_size 为元组
    if isinstance(pool_size, int):
        pool_size = (pool_size, pool_size)
    pool_h, pool_w = pool_size
    
    # 统一 stride
    if stride is None:
        stride = pool_size
    if isinstance(stride, int):
        stride = (stride, stride)
    stride_h, stride_w = stride
    
    # 统一 padding
    if isinstance(padding, int):
        padding = (padding, padding)
    pad_h, pad_w = padding
    
    # 处理输入维度：统一转为 (N, C, H, W)
    original_ndim = X.ndim
    if original_ndim == 2:
        X = X[np.newaxis, np.newaxis, :, :]
    elif original_ndim == 3:
        X = X[np.newaxis, :, :, :]
    
    N, C, H, W = X.shape
    
    # 计算输出尺寸
    H_out = (H + 2 * pad_h - pool_h) // stride_h + 1
    W_out = (W + 2 * pad_w - pool_w) // stride_w + 1
    
    # 判断输入数据类型，选择合适的填充值
    if np.issubdtype(X.dtype, np.integer):
        pad_value = np.iinfo(X.dtype).min
    else:
        pad_value = -np.inf
    
    # 对输入进行 padding
    X_padded = np.pad(
        X,
        ((0, 0), (0, 0), (pad_h, pad_h), (pad_w, pad_w)),
        mode='constant',
        constant_values=pad_value
    )
    
    # 初始化输出
    out = np.zeros((N, C, H_out, W_out), dtype=X.dtype)
    
    # 执行池化操作
    for i in range(H_out):
        for j in range(W_out):
            h_start = i * stride_h
            w_start = j * stride_w
            
            window = X_padded[:, :, h_start:h_start+pool_h, w_start:w_start+pool_w]
            out[:, :, i, j] = np.max(window, axis=(2, 3))
    
    # 恢复原始维度
    if original_ndim == 2:
        out = out[0, 0]
    elif original_ndim == 3:
        out = out[0]
    
    return out


# ============================================
# 测试代码
# ============================================
print("=" * 60)
print("2.2 手动实现 Max Pooling 测试")
print("=" * 60)

# 测试1：基本功能
np.random.seed(42)
X1 = np.random.randint(0, 10, (1, 2, 4, 4))
print(f"\n测试1：输入形状 {X1.shape}，pool_size=2，stride=2，padding=0")
print(f"输入 X1:\n{X1[0, 0]}")
out1 = max_pool2d(X1, pool_size=2, stride=2, padding=0)
print(f"输出形状: {out1.shape}")
print(f"输出:\n{out1[0, 0]}")

# 测试2：带 padding
X2 = np.random.randint(0, 10, (1, 1, 4, 4))
print(f"\n测试2：输入形状 {X2.shape}，pool_size=3，stride=2，padding=1")
print(f"输入 X2:\n{X2[0, 0]}")
out2 = max_pool2d(X2, pool_size=3, stride=2, padding=1)
print(f"输出形状: {out2.shape}")
print(f"输出:\n{out2[0, 0]}")

# 测试3：与 PyTorch 对比
print(f"\n测试3：与 PyTorch MaxPool2d 对比")
try:
    import torch
    X_torch = torch.tensor(X1, dtype=torch.float32)
    pool_torch = torch.nn.MaxPool2d(kernel_size=2, stride=2, padding=0)
    out_torch = pool_torch(X_torch)
    out_ours = max_pool2d(X1.astype(np.float32), pool_size=2, stride=2, padding=0)
    
    diff = np.abs(out_ours - out_torch.numpy())
    print(f"手动实现与 PyTorch 的最大差异: {diff.max():.6f}")
    print(f"两者完全一致: {np.allclose(out_ours, out_torch.numpy())}")
except ImportError:
    print("未安装 PyTorch，跳过对比验证")

# 测试4：不同输入维度
print(f"\n测试4：不同输入维度")
X2d = np.random.randn(4, 4)
out_2d = max_pool2d(X2d, pool_size=2, stride=2)
print(f"2D 输入 {X2d.shape} -> 输出 {out_2d.shape}")

X3d = np.random.randn(3, 4, 4)
out_3d = max_pool2d(X3d, pool_size=2, stride=2)
print(f"3D 输入 {X3d.shape} -> 输出 {out_3d.shape}")

X4d = np.random.randn(2, 3, 8, 8)
out_4d = max_pool2d(X4d, pool_size=2, stride=2, padding=1)
print(f"4D 输入 {X4d.shape} -> 输出 {out_4d.shape}")

print("\n" + "=" * 60)
print("所有测试完成！")

2.2 手动实现 Max Pooling 测试

测试1：输入形状 (1, 2, 4, 4)，pool_size=2，stride=2，padding=0
输入 X1:
[[6 3 7 4]
 [6 9 2 6]
 [7 4 3 7]
 [7 2 5 4]]
输出形状: (1, 2, 2, 2)
输出:
[[9 7]
 [7 7]]

测试2：输入形状 (1, 1, 4, 4)，pool_size=3，stride=2，padding=1
输入 X2:
[[4 2 6 4]
 [8 6 1 3]
 [8 1 9 8]
 [9 4 1 3]]
输出形状: (1, 1, 2, 2)
输出:
[[8 6]
 [9 9]]

测试3：与 PyTorch MaxPool2d 对比
手动实现与 PyTorch 的最大差异: 0.000000
两者完全一致: True

测试4：不同输入维度
2D 输入 (4, 4) -> 输出 (2, 2)
3D 输入 (3, 4, 4) -> 输出 (3, 2, 2)
4D 输入 (2, 3, 8, 8) -> 输出 (2, 3, 5, 5)

所有测试完成！


### 3.1 VGG 卷积参数量计算

假设输入和输出通道数均为 C，不带偏置。

---

#### 1. 一个 5×5 卷积层的参数量

每个输出通道需要一个 5×5 卷积核，覆盖所有 C 个输入通道。
单个卷积核参数量 = C × 5 × 5 = 25C
共有 C 个输出通道。

总参数量 = C × 25C = 25C²

答案：25C²

---

#### 2. 两个串联 3×3 卷积层的总参数量

第一层（C → C，3×3）：
参数量 = C × C × 3 × 3 = 9C²

第二层（C → C，3×3）：
参数量 = C × C × 3 × 3 = 9C²

总参数量 = 9C² + 9C² = 18C²

答案：18C²

---

#### 对比总结：

- 一个 5×5 卷积：25C² 参数
- 两个 3×3 卷积串联：18C² 参数

两个 3×3 卷积的感受野等价于一个 5×5 卷积，
但参数量减少 28%，同时增加了一层非线性激活，
这就是 VGG 网络用小卷积核级联替代大卷积核的动机。

In [4]:
import torch
import torch.nn as nn

class NiNBlock(nn.Module):
    """
    NiN 块：一个普通卷积层 + 两个 1×1 卷积层，每层后跟 ReLU
    """
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0):
        super(NiNBlock, self).__init__()
        
        self.block = nn.Sequential(
            # 普通卷积层
            nn.Conv2d(in_channels, out_channels, kernel_size, stride=stride, padding=padding),
            nn.ReLU(),
            # 第一个 1×1 卷积
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU(),
            # 第二个 1×1 卷积
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU()
        )
    
    def forward(self, x):
        return self.block(x)


# ============================================
# 测试代码
# ============================================
print("=" * 60)
print("3.2 NiN 块测试")
print("=" * 60)

# 创建 NiN 块
nin_block = NiNBlock(in_channels=3, out_channels=64, kernel_size=3, stride=1, padding=1)
print(f"\nNiN 块结构:\n{nin_block}")

# 统计参数量
total_params = sum(p.numel() for p in nin_block.parameters())
trainable_params = sum(p.numel() for p in nin_block.parameters() if p.requires_grad)
print(f"\n总参数量: {total_params:,}")
print(f"可训练参数量: {trainable_params:,}")

# 逐层参数量
print(f"\n逐层参数量：")
for name, param in nin_block.named_parameters():
    if 'weight' in name:
        print(f"  {name}: {param.shape} -> {param.numel():,}")

# 前向传播测试
x = torch.randn(4, 3, 32, 32)  # batch=4, channel=3, 32x32
print(f"\n输入形状: {x.shape}")
y = nin_block(x)
print(f"输出形状: {y.shape}")

print(f"\n输入 -> 输出维度变化：{x.shape} -> {y.shape}")
print("=" * 60)

3.2 NiN 块测试

NiN 块结构:
NiNBlock(
  (block): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1))
    (3): ReLU()
    (4): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1))
    (5): ReLU()
  )
)

总参数量: 10,112
可训练参数量: 10,112

逐层参数量：
  block.0.weight: torch.Size([64, 3, 3, 3]) -> 1,728
  block.2.weight: torch.Size([64, 64, 1, 1]) -> 4,096
  block.4.weight: torch.Size([64, 64, 1, 1]) -> 4,096

输入形状: torch.Size([4, 3, 32, 32])
输出形状: torch.Size([4, 64, 32, 32])

输入 -> 输出维度变化：torch.Size([4, 3, 32, 32]) -> torch.Size([4, 64, 32, 32])


### 4.1 批量归一化计算

已知：x1=2, x2=4, x3=6, x4=8，γ=2, β=1, ε=0

---

#### 第1步：计算均值 μ

μ = (2 + 4 + 6 + 8) / 4 = 20 / 4 = 5

---

#### 第2步：计算方差 σ²

σ² = [(2-5)² + (4-5)² + (6-5)² + (8-5)²] / 4
   = [(-3)² + (-1)² + 1² + 3²] / 4
   = [9 + 1 + 1 + 9] / 4
   = 20 / 4
   = 5

---

#### 第3步：标准化 x̂i = (xi - μ) / √(σ² + ε)

x̂1 = (2 - 5) / √5 = -3/√5
x̂2 = (4 - 5) / √5 = -1/√5
x̂3 = (6 - 5) / √5 =  1/√5
x̂4 = (8 - 5) / √5 =  3/√5

---

#### 第4步：缩放和平移 yi = γ·x̂i + β

y1 = 2 × (-3/√5) + 1 = 1 - 6/√5 ≈ 1 - 2.683 ≈ -1.683
y2 = 2 × (-1/√5) + 1 = 1 - 2/√5 ≈ 1 - 0.894 ≈  0.106
y3 = 2 × (1/√5) + 1  = 1 + 2/√5 ≈ 1 + 0.894 ≈  1.894
y4 = 2 × (3/√5) + 1  = 1 + 6/√5 ≈ 1 + 2.683 ≈  3.683

---

答案：
y1 ≈ -1.683, y2 ≈ 0.106, y3 ≈ 1.894, y4 ≈ 3.683

In [5]:
import torch
import torch.nn as nn

class ResidualBlock(nn.Module):
    """
    残差块：两个 3×3 卷积 + BN + ReLU，支持 1×1 shortcut
    """
    def __init__(self, in_channels, out_channels, use_1x1conv=False, stride=1):
        super(ResidualBlock, self).__init__()
        
        # 第一个卷积层
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, 
                               stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        
        # 第二个卷积层
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                               stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        self.relu = nn.ReLU(inplace=True)
        
        # shortcut 连接
        if use_1x1conv:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, 
                         stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
        else:
            self.shortcut = nn.Identity()
    
    def forward(self, x):
        # 主路径
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        
        out = self.conv2(out)
        out = self.bn2(out)
        
        # shortcut 连接
        shortcut = self.shortcut(x)
        
        # 相加后激活
        out = out + shortcut
        out = self.relu(out)
        
        return out


# ============================================
# 测试代码
# ============================================
print("=" * 60)
print("4.2 残差块测试")
print("=" * 60)

# 测试1：不改变通道数，不需要 1x1conv
print("\n测试1：通道数相同，use_1x1conv=False")
block1 = ResidualBlock(in_channels=64, out_channels=64, use_1x1conv=False)
x1 = torch.randn(4, 64, 32, 32)
y1 = block1(x1)
print(f"  输入形状: {x1.shape}")
print(f"  输出形状: {y1.shape}")
print(f"  输入输出形状一致: {x1.shape == y1.shape}")
print(f"  总参数量: {sum(p.numel() for p in block1.parameters()):,}")

# 测试2：改变通道数，需要 1x1conv
print("\n测试2：通道数改变，use_1x1conv=True")
block2 = ResidualBlock(in_channels=64, out_channels=128, use_1x1conv=True)
x2 = torch.randn(4, 64, 32, 32)
y2 = block2(x2)
print(f"  输入形状: {x2.shape}")
print(f"  输出形状: {y2.shape}")
print(f"  总参数量: {sum(p.numel() for p in block2.parameters()):,}")

# 测试3：改变通道数，不使用 1x1conv（会报错）
print("\n测试3：通道数改变但 use_1x1conv=False（验证会报错）")
block3 = ResidualBlock(in_channels=64, out_channels=128, use_1x1conv=False)
x3 = torch.randn(4, 64, 32, 32)
try:
    y3 = block3(x3)
    print("  没有报错（可能因为 stride=1 且形状巧合）")
except RuntimeError as e:
    print(f"  正确报错：形状不匹配")

# 测试4：带 stride 下采样
print("\n测试4：stride=2 下采样")
block4 = ResidualBlock(in_channels=64, out_channels=128, use_1x1conv=True, stride=2)
x4 = torch.randn(4, 64, 32, 32)
y4 = block4(x4)
print(f"  输入形状: {x4.shape}")
print(f"  输出形状: {y4.shape}")

# 测试5：梯度流验证
print("\n测试5：梯度流验证")
block5 = ResidualBlock(in_channels=64, out_channels=64)
x5 = torch.randn(4, 64, 32, 32, requires_grad=True)
y5 = block5(x5)
loss = y5.sum()
loss.backward()
print(f"  输入梯度不为 None: {x5.grad is not None}")
print(f"  conv1 权重梯度范数: {block5.conv1.weight.grad.norm().item():.6f}")
print(f"  conv2 权重梯度范数: {block5.conv2.weight.grad.norm().item():.6f}")

print("\n" + "=" * 60)
print("所有测试完成！")

4.2 残差块测试

测试1：通道数相同，use_1x1conv=False
  输入形状: torch.Size([4, 64, 32, 32])
  输出形状: torch.Size([4, 64, 32, 32])
  输入输出形状一致: True
  总参数量: 73,984

测试2：通道数改变，use_1x1conv=True
  输入形状: torch.Size([4, 64, 32, 32])
  输出形状: torch.Size([4, 128, 32, 32])
  总参数量: 230,144

测试3：通道数改变但 use_1x1conv=False（验证会报错）
  正确报错：形状不匹配

测试4：stride=2 下采样
  输入形状: torch.Size([4, 64, 32, 32])
  输出形状: torch.Size([4, 128, 16, 16])

测试5：梯度流验证
  输入梯度不为 None: True
  conv1 权重梯度范数: 17169.220703
  conv2 权重梯度范数: 16092.304688

所有测试完成！


### 5.1 微调理论

---

#### 1. 为什么底层特征提取层设置较小学习率，顶层输出层设置较大学习率？

原因分析：

（1）底层特征具有通用性：
    预训练模型在大型数据集（如 ImageNet）上学到的底层特征
    （边缘、纹理、形状等）对于大多数视觉任务是通用的。
    这些特征提取能力不应被大幅度改变。

（2）顶层特征具有任务特异性：
    顶层接近输出的层学习的是与原始任务（如1000类分类）相关的
    特定特征组合，当目标任务不同时，这些层需要更多调整。

（3）防止灾难性遗忘：
    如果对所有层使用大学习率，模型会快速忘记预训练学到的
    有用特征表示，导致预训练效果大打折扣。

（4）训练稳定性：
    底层参数如果变化过大，会逐层放大到顶层，导致训练不稳定。

策略：
- 底层特征提取层：使用较小的学习率（如 lr/10），或直接冻结
- 顶层/新初始化的输出层：使用正常或较大的学习率
- 实践中常使用分层学习率设置

---

#### 2. 目标数据集非常小且与源数据集非常相似时的微调策略

核心原则：避免过拟合，同时保留预训练的强大特征表示。

策略：

（1）冻结所有特征提取层：
    只训练最后的新分类层（全连接层），保持底层特征提取器的
    参数不变。因为数据少，微调整个网络极易过拟合。

（2）使用线性分类器：
    直接在预训练模型提取的特征上训练一个线性 SVM 或
    逻辑回归分类器，而不是微调整个网络。

（3）采用较小的学习率：
    即使只训练顶层，也使用较小的学习率，防止参数更新幅度过大。

（4）使用权重衰减（L2 正则化）：
    对最后分类层添加权重衰减，限制参数大小。

（5）使用 Dropout：
    在分类层之前添加 Dropout，增强泛化能力。

（6）数据增强：
    利用图像增广（旋转、翻转、裁剪等）从少量数据中
    创造更多训练样本。

（7）早停法：
    密切监控验证集性能，一旦开始下降立即停止训练。

总结：当目标数据集小且与源数据集相似时，最安全的策略是
     将预训练网络作为固定特征提取器，只训练顶部分类层。

In [6]:
import torch
import torchvision.transforms as transforms
import numpy as np
from PIL import Image

# ============================================
# 创建图像增广 Pipeline
# ============================================
transform_pipeline = transforms.Compose([
    transforms.RandomResizedCrop(size=224, scale=(0.08, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),
    transforms.ToTensor()
])

print("=" * 60)
print("5.2 图像增广 Pipeline 测试")
print("=" * 60)

# 生成测试图像
np.random.seed(42)
test_image = np.random.randint(0, 256, (224, 224, 3), dtype=np.uint8)
test_image_pil = Image.fromarray(test_image)

# 应用变换 3 次
print(f"\n测试图像大小: {test_image_pil.size}")

for i in range(3):
    augmented = transform_pipeline(test_image_pil)
    print(f"增广样本 {i+1}: 形状 {augmented.shape}, 值范围 [{augmented.min():.4f}, {augmented.max():.4f}]")

# 验证 Pipeline 结构
print(f"\nPipeline 结构：")
print(transform_pipeline)

# 验证输出是正确的图像格式
augmented_sample = transform_pipeline(test_image_pil)
print(f"\n输出形状: {augmented_sample.shape}")
print(f"数据类型: {augmented_sample.dtype}")
print(f"通道数: {augmented_sample.shape[0]}, 高: {augmented_sample.shape[1]}, 宽: {augmented_sample.shape[2]}")

print("\n" + "=" * 60)
print("Pipeline 创建完成！")

5.2 图像增广 Pipeline 测试

测试图像大小: (224, 224)
增广样本 1: 形状 torch.Size([3, 224, 224]), 值范围 [0.0000, 0.9255]
增广样本 2: 形状 torch.Size([3, 224, 224]), 值范围 [0.0000, 0.7647]
增广样本 3: 形状 torch.Size([3, 224, 224]), 值范围 [0.0000, 1.0000]

Pipeline 结构：
Compose(
    RandomResizedCrop(size=(224, 224), scale=(0.08, 1.0), ratio=(0.75, 1.3333), interpolation=bilinear, antialias=True)
    RandomHorizontalFlip(p=0.5)
    ColorJitter(brightness=(0.5, 1.5), contrast=(0.5, 1.5), saturation=(0.5, 1.5), hue=None)
    ToTensor()
)

输出形状: torch.Size([3, 224, 224])
数据类型: torch.float32
通道数: 3, 高: 224, 宽: 224

Pipeline 创建完成！


### 6.1 IoU（交并比）计算

已知：
真实框 A = [10, 10, 50, 50]
预测框 B = [30, 30, 70, 70]

---

#### 第1步：计算交集

交集左上角：x = max(10, 30) = 30
             y = max(10, 30) = 30
交集右下角：x = min(50, 70) = 50
             y = min(50, 70) = 50

交集面积 = (50 - 30) × (50 - 30) = 20 × 20 = 400

---

#### 第2步：各框面积

A 面积 = (50 - 10) × (50 - 10) = 40 × 40 = 1600
B 面积 = (70 - 30) × (70 - 30) = 40 × 40 = 1600

---

#### 第3步：并集面积

并集面积 = 1600 + 1600 - 400 = 2800

---

#### 第4步：IoU

IoU = 交集 / 并集 = 400 / 2800 = 1/7 ≈ 0.1429

答案：IoU = 1/7 ≈ 14.29%

In [7]:
import torch
import torch.nn.functional as F

def label_smoothing_cross_entropy(logits, targets, epsilon=0.1):
    """
    带标签平滑的交叉熵损失
    
    参数:
    - logits: 模型输出（未经过 softmax），形状 (N, K)
    - targets: 真实标签索引，形状 (N,)
    - epsilon: 平滑因子
    
    返回:
    - loss: 标签平滑后的交叉熵损失
    """
    K = logits.shape[1]
    
    # 创建平滑标签
    with torch.no_grad():
        smooth_labels = torch.full_like(logits, epsilon / (K - 1))
        smooth_labels.scatter_(1, targets.unsqueeze(1), 1 - epsilon)
    
    # 计算 log_softmax 和损失
    log_probs = F.log_softmax(logits, dim=1)
    loss = -(smooth_labels * log_probs).sum(dim=1).mean()
    
    return loss


# ============================================
# 测试
# ============================================
torch.manual_seed(42)

batch_size = 4
num_classes = 10
epsilon = 0.1

logits = torch.randn(batch_size, num_classes)
targets = torch.tensor([2, 5, 1, 8])

# 计算标签平滑损失
loss_smooth = label_smoothing_cross_entropy(logits, targets, epsilon)

# 标准交叉熵（对比）
loss_standard = F.cross_entropy(logits, targets)

print("=" * 60)
print("6.2 标签平滑交叉熵")
print("=" * 60)
print(f"类别数 K = {num_classes}")
print(f"平滑因子 epsilon = {epsilon}")
print(f"真实标签: {targets.tolist()}")
print(f"\n标签平滑后，真实类别概率: {1 - epsilon}")
print(f"标签平滑后，其他类别概率: {epsilon / (num_classes - 1):.4f}")
print(f"\n标签平滑交叉熵损失: {loss_smooth:.4f}")
print(f"标准交叉熵损失:     {loss_standard:.4f}")
print(f"损失差异:           {loss_smooth - loss_standard:.4f}")
print("=" * 60)

6.2 标签平滑交叉熵
类别数 K = 10
平滑因子 epsilon = 0.1
真实标签: [2, 5, 1, 8]

标签平滑后，真实类别概率: 0.9
标签平滑后，其他类别概率: 0.0111

标签平滑交叉熵损失: 2.6417
标准交叉熵损失:     2.6139
损失差异:           0.0278
